### Import and configs

In [3]:
import json
import glob
import pandas as pd
from pathlib import Path

### Parse all the json

In [ ]:
# Find all the json files
files = glob.glob('../data/api_raw/search/*.json')

# Iterate and parse   
rows = []

for file in files:
    with open(file, 'r', encoding='utf-8') as f:
        data = json.load(f)
        
    # Extract category and page from filename
    parts = Path(file).stem.split('_page_')
    
    # if not len == 2 means that another JSON is not from a searched product 
    if len(parts)==2:    
        category = parts[0]
        page     = int(parts[1])
    
        search_results = data.get('search_results', [])
        if not search_results:
            #print(f'No results: {file}')
            continue

        for item in search_results:
            prices = item.get("prices", [])
            price_primary = prices[0].get("value") if len(prices) > 0 else None
            price_rrp     = prices[1].get("value") if len(prices) > 1 else None                

            is_sponsored = item.get('sponsored', False)
                
            rows.append({
                'asin':           item.get('asin'),
                'title':          item.get('title'),
                'rating':         item.get('rating'),
                'review_count':   item.get('ratings_total'),
                'sponsored':      is_sponsored,
                'is_prime':       item.get('is_prime'),
                'search_position':item.get('position'),
                'recent_sales':   item.get('recent_sales'),
                'price_primary':  price_primary,
                'price_rrp':      price_rrp,
                'category':       category,
                'page':           page,
            })
    else:
        print(f"[SKIP] {Path(file).stem}")

df = pd.DataFrame(rows)
df.head(10)

,asin,title,rating,review_count,sponsored,is_prime,search_position,recent_sales,price_primary,price_rrp,category,page
0,B09NXS395V,24K Gold Under Eye Patches - 60 Pack for Puffy...,4.1,14615.0,False,False,1,10K+ bought in past month,9.39,9.99,beauty_personal_care,1
1,B091NJQ29P,Good Molecules Yerba Mate Wake Up Eye Gel - Hy...,4.2,28653.0,False,False,2,60K+ bought in past month,5.97,NaN,beauty_personal_care,1
2,B08Z5X1L1S,Gifts for Women Gift Basket for Women - 10 Pc ...,4.6,2369.0,False,False,3,5K+ bought in past month,34.98,65.00,beauty_personal_care,1
3,B08KT2Z93D,"eos Shea Better Body Lotion Vanilla Cashmere, ...",4.7,65741.0,False,False,4,100K+ bought in past month,9.97,10.99,beauty_personal_care,1
4,B0CMV2DCXT,CHMI Under Eye Patches (50 Pairs) - 24K Gold E...,4.5,1021.0,False,False,5,5K+ bought in past month,9.99,NaN,beauty_personal_care,1
5,B0CTK5ZTNK,"Under Eye Patches, 40 Pairs Eye Mask for Dark ...",4.5,5192.0,False,False,6,10K+ bought in past month,9.99,12.99,beauty_personal_care,1
6,B0CNV5SG4S,"Sleeping lip mask, Nourish & Hydrate Lip Mask ...",4.6,969.0,False,False,7,300+ bought in past month,4.99,6.99,beauty_personal_care,1
7,B0C7W6GW8P,"e.l.f. Squeeze Me Lip Balm, Moisturizing Lip B...",4.5,19260.0,False,False,8,10K+ bought in past month,5.00,NaN,beauty_personal_care,1
8,B0CPCYP629,eos 24H Moisture Travel Body Lotion- Vanilla C...,4.8,3099.0,False,False,9,20K+ bought in past month,3.99,NaN,beauty_personal_care,1
9,B0DMTDN158,The Ordinary Glycolic Acid 7% Exfoliating Tone...,4.7,44753.0,False,False,10,40K+ bought in past month,7.65,9.00,beauty_personal_care,1


### Dataframe of products search, create CSV


In [5]:
df.to_csv('../data/processed/products_search.csv', index=False)
print(f"Saved: {df.shape}")

Saved: (1386, 12)


## Best Products by Total Sales
We need to convert recent_sales to Integer for evaluate with a sort by sales, and collect the top 8 products by each category

### Function to parse recent sales to a number

In [21]:
def parse_recent_sales(value):
    # get the integer value from recent sales
    if not isinstance(value, str):
        return None
    
    value = value.upper()

    if 'K+' in value:
        return int(float(value.split('K+')[0].strip()) *1000)
    elif '+' in value:
        return int(value.split('+')[0].strip())
    else:
        return None
        

### Applied function to cleanup

In [23]:
df['recent_sales_approx'] = df['recent_sales'].apply(parse_recent_sales)
df.head()

,asin,title,rating,review_count,sponsored,is_prime,search_position,recent_sales,price_primary,price_rrp,category,page,recent_sales_approx
0,B09NXS395V,24K Gold Under Eye Patches - 60 Pack for Puffy...,4.1,14615.0,False,False,1,10K+ bought in past month,9.39,9.99,Beauty Personal Care,1,10000.0
1,B091NJQ29P,Good Molecules Yerba Mate Wake Up Eye Gel - Hy...,4.2,28653.0,False,False,2,60K+ bought in past month,5.97,NaN,Beauty Personal Care,1,60000.0
2,B08Z5X1L1S,Gifts for Women Gift Basket for Women - 10 Pc ...,4.6,2369.0,False,False,3,5K+ bought in past month,34.98,65.00,Beauty Personal Care,1,5000.0
3,B08KT2Z93D,"eos Shea Better Body Lotion Vanilla Cashmere, ...",4.7,65741.0,False,False,4,100K+ bought in past month,9.97,10.99,Beauty Personal Care,1,100000.0
4,B0CMV2DCXT,CHMI Under Eye Patches (50 Pairs) - 24K Gold E...,4.5,1021.0,False,False,5,5K+ bought in past month,9.99,NaN,Beauty Personal Care,1,5000.0


### Top 8 products each category by ASIN

In [ ]:
top_asins = (
    df.dropna(subset=['recent_sales_approx'])
    .sort_values('recent_sales_approx', ascending=False)
    .drop_duplicates(subset='asin')
    .groupby('category')
    .head(8)['asin']
    .tolist()
)

print(f"ASINs to query: {len(top_asins)}")
top_asins
df[df['asin'].isin(top_asins)].sort_values('recent_sales_approx', ascending=False)

ASINs to query: 32


,asin,title,rating,review_count,sponsored,is_prime,search_position,recent_sales,price_primary,price_rrp,category,page,recent_sales_approx
3,B08KT2Z93D,"eos Shea Better Body Lotion Vanilla Cashmere, ...",4.7,65741.0,False,False,4,100K+ bought in past month,9.97,10.99,beauty_personal_care,1,100000.0
32,B09541P9WH,Amazon Basics Double-Tipped Cotton Swabs for P...,4.7,71667.0,False,False,33,100K+ bought in past month,2.84,NaN,beauty_personal_care,1,100000.0
38,B0D8W1YVBX,EQQUALBERRY Vitamin Illuminating Serum | Niaci...,4.4,8033.0,False,False,39,100K+ bought in past month,19.99,24.99,beauty_personal_care,1,100000.0
26,B0DBF65JYY,medicube PDRN Pink Peptide Serum with Salmon D...,4.6,10387.0,False,False,27,80K+ bought in past month,18.90,21.80,beauty_personal_care,1,80000.0
74,B00AHAWWO0,Crest 3D Whitestrips Professional Effects – Te...,4.6,102791.0,False,False,22,80K+ bought in past month,45.99,NaN,beauty_personal_care,2,80000.0
19,B0DPHQRLJC,"eos Cashmere Body Wash – Vanilla Cashmere, Moi...",4.8,14386.0,False,False,20,70K+ bought in past month,9.98,11.99,beauty_personal_care,1,70000.0
1,B091NJQ29P,Good Molecules Yerba Mate Wake Up Eye Gel - Hy...,4.2,28653.0,False,False,2,60K+ bought in past month,5.97,NaN,beauty_personal_care,1,60000.0
342,B0BK2SC18T,GuruNanda Teeth Whitening Strips - 7-Day Treat...,4.4,17686.0,False,False,20,60K+ bought in past month,9.97,NaN,beauty_personal_care,7,60000.0
772,B0113UZJE2,"Etekcity Food Kitchen Scale, Digital Grams and...",4.6,144208.0,False,False,3,50K+ bought in past month,13.99,NaN,home_kitchen,3,50000.0
682,B0CJ1B6D6S,Scrub Daddy Scrub Mommy Sponges - Dish Scrubbe...,4.8,31676.0,False,False,31,30K+ bought in past month,13.99,NaN,home_kitchen,1,30000.0


### Saves ASINs in a JSON file

In [ ]:
with open('../data/api_raw/top_asins.json', 'w') as f:
    json.dump(top_asins, f)

print(f"Saved {len(top_asins)} ASINs")

Saved 32 ASINs
